In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Unzip the Dataset
!unzip -n "/content/drive/MyDrive/Maize_seed_dataset/GOOD_SEED.zip" -d "/content/drive/MyDrive/MaizeData/"

In [ ]:
# Unzip the Dataset
!unzip -n "/content/drive/MyDrive/Maize_seed_dataset/BAD_SEED.zip" -d "/content/drive/MyDrive/MaizeData/"

In [ ]:
# Unzip the Dataset
!unzip -n "/content/drive/MyDrive/Maize_seed_dataset/NO_MAIZE.zip" -d "/content/drive/MyDrive/MaizeData/"

In [ ]:
!pip install -q pillow-heif

In [ ]:
from pathlib import Path
from PIL import Image, ImageOps
from pillow_heif import register_heif_opener

register_heif_opener()

# Change only this line if your folder has a different location
ROOT_DIR = Path("/content/drive/MyDrive/MaizeData")

JPEG_QUALITY = 95
DELETE_HEIC_AFTER_CONVERSION = False   # Keep originals for safety

heic_files = [
    p for p in ROOT_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in {".heic", ".heif"}
]

print(f"Found {len(heic_files)} HEIC/HEIF images.")

converted = 0
failed = 0
skipped = 0

for heic_path in heic_files:

    jpg_path = heic_path.with_suffix(".jpg")

    # Don't reconvert an image if its JPG already exists
    if jpg_path.exists():
        skipped += 1
        continue

    try:
        with Image.open(heic_path) as img:

            # Correct iPhone portrait/landscape orientation
            img = ImageOps.exif_transpose(img)

            # JPG requires RGB
            img = img.convert("RGB")

            img.save(
                jpg_path,
                "JPEG",
                quality=JPEG_QUALITY,
                optimize=True
            )

        converted += 1

        if DELETE_HEIC_AFTER_CONVERSION:
            heic_path.unlink()

    except Exception as e:
        failed += 1
        print(f"FAILED: {heic_path}")
        print(f"Reason: {e}")

print("\nConversion complete.")
print(f"Converted : {converted}")
print(f"Skipped   : {skipped}")
print(f"Failed    : {failed}")

# Maize Seed Identification & Quality Classification — MobileNetV3Large

This notebook trains a **3-class classifier** for:

- `BAD_SEED`
- `GOOD_SEED`
- `NO_MAIZE`

It is designed for Google Colab and a Raspberry Pi deployment workflow. Once the three folders contain your raw camera images, use **Runtime → Run all**.

## Expected folder structure

```text
/content/drive/MyDrive/MaizeData/
├── BAD_SEED/
├── GOOD_SEED/
└── NO_MAIZE/
```

Use real camera images with the same background, distance and lighting the deployed sorter will see. Add the exact objects/patterns that previously fooled the model to `NO_MAIZE`. After splitting the real images, the notebook saves enough augmented training images to reach **3,000 per class (9,000 total)**. Validation and test sets contain only held-out originals.

## Before Run all

The notebook writes a separate timestamped output folder for each run. It keeps
original photos, excludes exact duplicates, and saves augmented images separately.
If multiple photos show the same physical seed or recording session, create
`MaizeData/source_groups.csv` before running:

```csv
relative_path,group_id
GOOD_SEED/photo_01.jpg,session_A
GOOD_SEED/photo_02.jpg,session_A
BAD_SEED/photo_03.jpg,session_B
```

Supply a row for every retained original. Use enough independent groups for all
three classes to occur in train, validation and test. Grouping reduces leakage;
a random split alone cannot detect different views of the same seed. Class labels
must describe defects visible in the image.

In [ ]:
# Mount Drive and import dependencies
import os, json, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, f1_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## Fixed class mapping

The mapping is deliberately fixed so it cannot silently change between training and deployment:

```text
0 = BAD_SEED
1 = GOOD_SEED
2 = NO_MAIZE
```

In [ ]:
# 2. Configuration
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
# Reproducibility within the same TensorFlow/hardware environment.
tf.config.experimental.enable_op_determinism()

DATA_ROOT = Path('/content/drive/MyDrive/MaizeData')
from datetime import datetime, timezone
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUTPUT_DIR = DATA_ROOT / 'model_outputs' / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ['BAD_SEED', 'GOOD_SEED', 'NO_MAIZE']
CLASS_TO_INDEX = {name: i for i, name in enumerate(CLASS_NAMES)}
INDEX_TO_CLASS = {i: name for name, i in CLASS_TO_INDEX.items()}

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
TARGET_TRAIN_IMAGES_PER_CLASS = 3000
AUGMENTED_ROOT = DATA_ROOT / 'augmented_training'
GROUPS_CSV = DATA_ROOT / 'source_groups.csv'  # Optional: relative_path,group_id
TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15

HEAD_EPOCHS = 18
FINETUNE_EPOCHS = 22
HEAD_LR = 1e-3
FINETUNE_LR = 1e-5
UNFREEZE_LAST_N_LAYERS = 70

print('DATA_ROOT:', DATA_ROOT)
print('CLASS_TO_INDEX:', CLASS_TO_INDEX)

In [ ]:
# 3. Decode fully, correct EXIF orientation, remove exact duplicates, normalize formats.
import hashlib
from PIL import ImageOps
SUPPORTED_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
PREPARED_ROOT = DATA_ROOT / 'prepared_images' / 'rgb_bilinear_v1_224'
PREPARED_ROOT.mkdir(parents=True, exist_ok=True)
records, corrupt_files, duplicates = [], [], []
seen = {}
for class_name in CLASS_NAMES:
    class_dir = DATA_ROOT / class_name
    if not class_dir.is_dir():
        raise FileNotFoundError(f'Missing folder: {class_dir}')
    paths = sorted(p for p in class_dir.rglob('*') if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS
                   and not any(part.startswith('.') or part == '__MACOSX' for part in p.relative_to(class_dir).parts))
    for path in tqdm(paths, desc=f'Preparing {class_name}'):
        try:
            with Image.open(path) as opened:
                img = ImageOps.exif_transpose(opened).convert('RGB')
                img.load()  # verify() alone can miss truncated pixel data.
                digest = hashlib.sha256(str(img.size).encode() + img.tobytes()).hexdigest()
                resized = img.resize((IMAGE_SIZE[1], IMAGE_SIZE[0]), Image.Resampling.BILINEAR)
        except (OSError, ValueError) as exc:
            corrupt_files.append({'path': str(path), 'reason': str(exc)})
            continue
        if digest in seen:
            previous = seen[digest]
            if previous['label_name'] != class_name:
                raise ValueError(f"Identical image has conflicting labels: {path} and {previous['source_path']}")
            duplicates.append({'path': str(path), 'kept_path': previous['source_path']})
            continue
        destination = PREPARED_ROOT / f'{digest}_{IMAGE_SIZE[0]}x{IMAGE_SIZE[1]}.png'
        # Lossless canonical originals give train, validation, test and Pi the same resize.
        resized.save(destination, format='PNG')
        row = {'path': str(destination), 'source_path': str(path), 'content_hash': digest,
               'label_name': class_name, 'label': CLASS_TO_INDEX[class_name]}
        seen[digest] = row
        records.append(row)
df = pd.DataFrame(records)
if df.empty:
    raise RuntimeError('No valid images found.')
counts = df['label_name'].value_counts().reindex(CLASS_NAMES, fill_value=0)
print('Unique usable originals per class:\n', counts)
if (counts < 10).any():
    raise ValueError('Each class needs at least 10 unique originals for a three-way split; collect more images.')
pd.DataFrame(corrupt_files, columns=['path', 'reason']).to_csv(OUTPUT_DIR/'unreadable_images.csv', index=False)
pd.DataFrame(duplicates, columns=['path', 'kept_path']).to_csv(OUTPUT_DIR/'duplicate_images.csv', index=False)
print(f'Excluded {len(corrupt_files)} unreadable files and {len(duplicates)} exact duplicates.')
print('Different photos of the same seed/session still need group IDs to stay in one split.')

In [ ]:
# 4. Split ORIGINALS before augmentation, optionally keeping capture groups together.
def choose_group_split(frame, test_size, seed):
    # Search reproducible group assignments for approximate class balance.
    splitter = GroupShuffleSplit(n_splits=200, test_size=test_size, random_state=seed)
    total = frame['label'].value_counts().reindex(range(3), fill_value=0)
    best = None
    for a, b in splitter.split(frame, frame['label'], groups=frame['group_id']):
        left, right = frame.iloc[a], frame.iloc[b]
        lc = left['label'].value_counts().reindex(range(3), fill_value=0)
        rc = right['label'].value_counts().reindex(range(3), fill_value=0)
        if (lc == 0).any() or (rc == 0).any():
            continue
        error = float(np.abs(rc / total - test_size).mean())
        if best is None or error < best[0]:
            best = (error, left.copy(), right.copy())
    if best is None:
        raise ValueError('Cannot split groups with all classes present. Add independent groups for every class.')
    return best[1], best[2]

if GROUPS_CSV.exists():
    groups = pd.read_csv(GROUPS_CSV, dtype=str, keep_default_na=False)
    if not {'relative_path', 'group_id'} <= set(groups.columns) or groups['relative_path'].duplicated().any():
        raise ValueError('source_groups.csv needs unique relative_path rows and a group_id column.')
    group_map = groups.set_index('relative_path')['group_id'].to_dict()
    df['group_id'] = df['source_path'].map(lambda p: group_map.get(Path(p).relative_to(DATA_ROOT).as_posix()))
    if df['group_id'].isna().any() or (df['group_id'].str.strip() == '').any():
        raise ValueError('Every retained original needs a nonempty group ID in source_groups.csv.')
    train_df, temp_df = choose_group_split(df, VAL_FRACTION + TEST_FRACTION, SEED)
    val_df, test_df = choose_group_split(temp_df, TEST_FRACTION / (VAL_FRACTION + TEST_FRACTION), SEED + 1)
else:
    print('No source_groups.csv: stratifying individual originals. Burst photos/same-seed photos can still leak.')
    df['group_id'] = df['content_hash']
    train_df, temp_df = train_test_split(df, test_size=VAL_FRACTION + TEST_FRACTION, random_state=SEED, stratify=df['label'])
    val_df, test_df = train_test_split(temp_df, test_size=TEST_FRACTION / (VAL_FRACTION + TEST_FRACTION), random_state=SEED, stratify=temp_df['label'])
for a, b in [(train_df, val_df), (train_df, test_df), (val_df, test_df)]:
    assert set(a['group_id']).isdisjoint(b['group_id'])
    assert set(a['content_hash']).isdisjoint(b['content_hash'])
for name, split_df in [('TRAIN ORIGINALS', train_df), ('VALIDATION', val_df), ('TEST', test_df)]:
    counts = split_df['label_name'].value_counts().reindex(CLASS_NAMES, fill_value=0)
    if (counts == 0).any():
        raise ValueError(f'Missing class in {name}')
    print(name, len(split_df), '\n', counts)
manifest = pd.concat([train_df.assign(split='train'), val_df.assign(split='validation'), test_df.assign(split='test')], ignore_index=True)
manifest.to_csv(OUTPUT_DIR/'dataset_split_manifest.csv', index=False)
raw_train_df = train_df.copy()

## Expand each training class to 3,000 images

Run this section after the original 70/15/15 split and before creating `train_ds`.
It writes **original training images plus augmented versions** into
`MaizeData/augmented_training/<dataset-id>/<class-name>/`, then points `train_df`
at those files. Each class reaches `TARGET_TRAIN_IMAGES_PER_CLASS` (default 3,000).
Classes already above the target keep all their training images.

Only training originals are used as augmentation sources. Validation and test
images stay unchanged. Flips, rotations up to 15 degrees, and brightness/contrast
changes of at most 10% preserve seed appearance; no hue changes or aggressive
crops are used. Saved images are RGB JPEGs at the model input size (224 × 224).

The seed and source content determine the output folder. Reruns reuse valid
files; changing the sources or configuration creates a separate folder. Originals
are never overwritten. `training_dataset_manifest.csv` records each image's
source and whether it was augmented. Use the split manifests for training;
do not combine generated images with originals and split them again.

Augmentation adds variations, not independent observations. In particular, keep
collecting real `NO_MAIZE` examples if that class has very few source images.


In [ ]:
# 4b. Save enough training-only variations to reach the target for every class.
import hashlib
from PIL import ImageEnhance, ImageOps


def expand_training_images(original_df, output_root, target, image_size, seed):
    if target < 1:
        raise ValueError('The per-class target must be positive.')
    for class_name in CLASS_NAMES:
        if not (original_df['label_name'] == class_name).any():
            raise ValueError(f'No original training images for {class_name}.')

    originals = original_df.sort_values(['label', 'path']).reset_index(drop=True)
    # Include content in the fingerprint so edited source photos invalidate the cache.
    recipe = {'version': 2, 'target': target, 'size': list(image_size), 'seed': seed}
    fingerprint = hashlib.sha256(json.dumps(recipe, sort_keys=True).encode())
    for row in tqdm(originals.itertuples(index=False), total=len(originals), desc='Fingerprinting originals'):
        fingerprint.update(json.dumps([row.label_name, row.path]).encode())
        with open(row.path, 'rb') as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b''):
                fingerprint.update(chunk)
    run_dir = Path(output_root) / fingerprint.hexdigest()[:20]
    run_dir.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(seed)
    height, width = image_size  # TensorFlow uses (height, width); Pillow uses (width, height).
    rows = []

    def save_if_needed(img, destination):
        if destination.exists():
            try:
                with Image.open(destination) as existing:
                    existing.load()
                    if existing.mode == 'RGB' and existing.size == (width, height):
                        return
            except (OSError, ValueError):
                pass  # Repair an incomplete image from an interrupted run.
        temporary = destination.with_suffix('.tmp')
        img.save(temporary, format='JPEG', quality=95)
        temporary.replace(destination)

    def variation(img, local_rng):
        if local_rng.random() < 0.5:
            img = ImageOps.mirror(img)
        if local_rng.random() < 0.5:
            img = ImageOps.flip(img)
        # Reflect padding avoids black corners without cropping away seed defects.
        pad = max(width, height) // 4 + 1
        padded = np.pad(np.asarray(img), ((pad, pad), (pad, pad), (0, 0)), mode='reflect')
        img = Image.fromarray(padded).rotate(
            float(local_rng.uniform(-15, 15)), resample=Image.Resampling.BICUBIC,
        ).crop((pad, pad, pad + width, pad + height))
        img = ImageEnhance.Brightness(img).enhance(float(local_rng.uniform(0.9, 1.1)))
        return ImageEnhance.Contrast(img).enhance(float(local_rng.uniform(0.9, 1.1)))

    for class_name in CLASS_NAMES:
        sources = originals[originals['label_name'] == class_name].reset_index(drop=True)
        needed = max(0, target - len(sources))
        # Spread variations evenly so no single source photo dominates a class.
        copies_per_source = np.full(len(sources), needed // len(sources), dtype=int)
        copies_per_source[rng.permutation(len(sources))[:needed % len(sources)]] += 1
        class_dir = run_dir / class_name
        class_dir.mkdir(exist_ok=True)
        for index, row in enumerate(tqdm(sources.itertuples(index=False), total=len(sources), desc=class_name)):
            with Image.open(row.path) as source_image:
                img = ImageOps.exif_transpose(source_image).convert('RGB')
                img = img.resize((width, height), Image.Resampling.BILINEAR)
            for variant in range(int(copies_per_source[index]) + 1):
                filename = f'original_{index:06d}.jpg' if variant == 0 else f'aug_{index:06d}_{variant:04d}.jpg'
                destination = class_dir / filename
                result = img if variant == 0 else variation(img, rng)
                save_if_needed(result, destination)
                rows.append({
                    'path': str(destination), 'label_name': class_name, 'label': int(row.label),
                    'source_path': row.path, 'original_path': row.source_path,
                    'group_id': row.group_id, 'is_augmented': variant > 0,
                })
        print(f'{class_name}: {len(sources)} originals + {needed} augmented = {len(sources) + needed}')

    expanded = pd.DataFrame(rows)
    for class_name in CLASS_NAMES:
        expected = max(target, int((originals['label_name'] == class_name).sum()))
        assert int((expanded['label_name'] == class_name).sum()) == expected
    return expanded, run_dir


train_df, AUGMENTED_TRAIN_DIR = expand_training_images(
    raw_train_df, AUGMENTED_ROOT, TARGET_TRAIN_IMAGES_PER_CLASS, IMAGE_SIZE, SEED,
)
held_out_paths = set(val_df['path']) | set(test_df['path'])
assert set(train_df['source_path']).isdisjoint(held_out_paths), 'Training source leaked into evaluation.'
training_manifest = pd.concat([
    train_df.assign(split='train'),
    val_df.assign(split='validation', source_path=val_df['path'], is_augmented=False),
    test_df.assign(split='test', source_path=test_df['path'], is_augmented=False),
], ignore_index=True)
training_manifest.to_csv(OUTPUT_DIR / 'training_dataset_manifest.csv', index=False)
training_manifest.to_csv(AUGMENTED_TRAIN_DIR / 'training_dataset_manifest.csv', index=False)
print('Training images saved in:', AUGMENTED_TRAIN_DIR)
print('Total training images:', len(train_df))

In [ ]:
# 5. Visual sanity check
fig, axes = plt.subplots(3, 5, figsize=(14, 8))
for row, class_name in enumerate(CLASS_NAMES):
    subset = train_df[train_df['label_name'] == class_name]
    samples = subset.sample(n=min(5, len(subset)), random_state=SEED)
    for col in range(5):
        ax = axes[row, col]
        ax.axis('off')
        if col < len(samples):
            img = Image.open(samples.iloc[col]['path']).convert('RGB')
            ax.imshow(img)
            ax.set_title(class_name)
plt.tight_layout()
plt.show()

## Consistent input preprocessing

All real images are fully decoded, EXIF-corrected, converted to RGB and resized
with **Pillow bilinear** before splitting. The Raspberry Pi uses that same method.
The exported model embeds MobileNet scaling and accepts `float32` RGB pixels in
`0..255`; do not normalize them externally. Use a fixed single-seed view matching
the training photos. A full-frame classifier does not count seeds or locate them.

In [ ]:
# 6. Stage on the Colab VM for faster training; use only canonical PNG/JPEG files.
import shutil
LOCAL_CACHE = Path('/content/maize_training_cache') / RUN_ID
LOCAL_CACHE.mkdir(parents=True, exist_ok=True)
def stage_split(frame, name):
    staged = frame.copy()
    folder = LOCAL_CACHE / name
    folder.mkdir(exist_ok=True)
    paths = []
    for i, source in enumerate(tqdm(frame.get('drive_path', frame['path']), desc=f'Staging {name}')):
        source = Path(source)
        destination = folder / f'{i:06d}{source.suffix}'
        shutil.copy2(source, destination)
        paths.append(str(destination))
    staged['drive_path'] = frame.get('drive_path', frame['path'])
    staged['path'] = paths
    return staged
train_df = stage_split(train_df, 'train')
val_df = stage_split(val_df, 'validation')
test_df = stage_split(test_df, 'test')
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    image = tf.ensure_shape(image, [IMAGE_SIZE[0], IMAGE_SIZE[1], 3])
    return tf.cast(image, tf.float32), tf.cast(label, tf.int32)

def make_dataset(split_df, training=False):
    ds = tf.data.Dataset.from_tensor_slices((split_df['path'].values, split_df['label'].astype(np.int32).values))
    if training:
        ds = ds.shuffle(len(split_df), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE, deterministic=True)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)
xb, yb = next(iter(train_ds))
print('Batch:', xb.shape, yb.shape, 'range:', float(tf.reduce_min(xb)), float(tf.reduce_max(xb)))

In [ ]:
# 7. Augmentation has already been saved by section 4b.
# Keep an identity wrapper so training does not stack a second set of transforms.
data_augmentation = keras.Sequential([
    layers.Activation('linear'),
], name='saved_augmentation_passthrough')


## Model choice: MobileNetV3Large

`MobileNetV3Large` is used as the accuracy-oriented MobileNet option while remaining practical for embedded deployment. Training is done in two stages: frozen ImageNet transfer learning, then low-learning-rate fine-tuning of the top backbone layers.

In [ ]:
# 8. Build MobileNetV3Large
base_model = tf.keras.applications.MobileNetV3Large(
    input_shape=IMAGE_SIZE + (3,),
    include_top=False,
    weights='imagenet',
    include_preprocessing=False,
)
base_model.trainable = False

# Core inference model: this is the model that will be exported to Raspberry Pi.
core_inputs = keras.Input(shape=IMAGE_SIZE + (3,), name='image')
x = layers.Rescaling(1.0 / 127.5, offset=-1.0, name='mobilenet_rescaling')(core_inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D(name='global_average_pool')(x)
x = layers.Dropout(0.35, name='dropout')(x)
core_outputs = layers.Dense(3, activation='softmax', kernel_regularizer=keras.regularizers.l2(1e-4), dtype='float32', name='class_probabilities')(x)
inference_model = keras.Model(core_inputs, core_outputs, name='maize_mobilenetv3large_core')

# Training wrapper consumes the images augmented and saved in section 4b.
train_inputs = keras.Input(shape=IMAGE_SIZE + (3,), name='training_image')
augmented = data_augmentation(train_inputs)
train_outputs = inference_model(augmented)
model = keras.Model(train_inputs, train_outputs, name='maize_mobilenetv3large_training')

model.summary()

In [ ]:
# 9. Balanced class weights
classes_present = np.array(sorted(train_df['label'].unique()))
weights = compute_class_weight('balanced', classes=classes_present, y=train_df['label'].values)
class_weights = {int(c): float(w) for c, w in zip(classes_present, weights)}
print({INDEX_TO_CLASS[k]: v for k, v in class_weights.items()})

In [ ]:
# 10. Monitor macro F1 so the smallest real validation class matters equally.
BEST_HEAD_PATH = OUTPUT_DIR/'best_head.keras'
BEST_FINETUNED_PATH = OUTPUT_DIR/'best_finetuned.keras'
class ValidationMacroF1(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        predictions = np.argmax(self.model.predict(val_ds, verbose=0), axis=1)
        score = float(f1_score(val_df['label'], predictions, labels=[0, 1, 2], average='macro', zero_division=0))
        if logs is not None:
            logs['val_macro_f1'] = score
        print(f' — val_macro_f1: {score:.4f}')

def callbacks_for(path):
    return [
        ValidationMacroF1(),
        keras.callbacks.ModelCheckpoint(str(path), monitor='val_macro_f1', mode='max', save_best_only=True, verbose=1),
        keras.callbacks.EarlyStopping(monitor='val_macro_f1', mode='max', patience=6, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.25, patience=2, min_lr=1e-7),
        keras.callbacks.CSVLogger(str(path.with_suffix('.history.csv'))),
        keras.callbacks.TerminateOnNaN(),
    ]

In [ ]:
# 11. Stage 1 — train classification head
model.compile(
    optimizer=keras.optimizers.Adam(HEAD_LR, clipnorm=1.0),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name='accuracy')],
)

history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=HEAD_EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks_for(BEST_HEAD_PATH),
)

In [ ]:
# 12. Stage 2 — fine-tune the top of MobileNetV3Large
# Explicitly start from the best head epoch, including when the epoch limit was reached.
head_checkpoint = keras.models.load_model(BEST_HEAD_PATH, compile=False)
model.set_weights(head_checkpoint.get_weights())
del head_checkpoint
base_model.trainable = True
for layer in base_model.layers[:-UNFREEZE_LAST_N_LAYERS]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

print('Trainable backbone layers:', sum(int(l.trainable) for l in base_model.layers), '/', len(base_model.layers))

model.compile(
    optimizer=keras.optimizers.Adam(FINETUNE_LR, clipnorm=1.0),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name='accuracy')],
)

history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINETUNE_EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks_for(BEST_FINETUNED_PATH),
)

In [ ]:
# 13. Plot training history
def merge_histories(*histories):
    out = {}
    for h in histories:
        for k, v in h.history.items(): out.setdefault(k, []).extend(v)
    return out

hist = merge_histories(history_head, history_finetune)

plt.figure(figsize=(8,5))
plt.plot(hist['accuracy'], label='train')
plt.plot(hist['val_accuracy'], label='validation')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.title('Accuracy'); plt.legend(); plt.grid(alpha=.25)
plt.tight_layout(); plt.savefig(OUTPUT_DIR/'training_accuracy.png', dpi=160); plt.show()

plt.figure(figsize=(8,5))
plt.plot(hist['loss'], label='train')
plt.plot(hist['val_loss'], label='validation')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Loss'); plt.legend(); plt.grid(alpha=.25)
plt.tight_layout(); plt.savefig(OUTPUT_DIR/'training_loss.png', dpi=160); plt.show()

In [ ]:
# 14. Final held-out test evaluation
# Choose using VALIDATION only; fine-tuning is not automatically better.
checkpoint_scores = []
for checkpoint in [BEST_HEAD_PATH, BEST_FINETUNED_PATH]:
    candidate = keras.models.load_model(checkpoint)
    candidate_loss = float(candidate.evaluate(val_ds, verbose=0)[0])
    candidate_pred = np.argmax(candidate.predict(val_ds, verbose=0), axis=1)
    macro_f1 = float(f1_score(val_df['label'], candidate_pred, labels=[0, 1, 2], average='macro', zero_division=0))
    if not np.isfinite(candidate_loss):
        raise RuntimeError(f'Nonfinite validation loss for {checkpoint}')
    checkpoint_scores.append({'checkpoint': str(checkpoint), 'macro_f1': macro_f1, 'val_loss': candidate_loss})
    del candidate
checkpoint_scores.sort(key=lambda row: (-row['macro_f1'], row['val_loss']))
final_checkpoint = checkpoint_scores[0]['checkpoint']
print('Checkpoint comparison:', checkpoint_scores)
model = keras.models.load_model(final_checkpoint)
inference_model = model.get_layer('maize_mobilenetv3large_core')

loss, acc = model.evaluate(test_ds, verbose=1)
print(f'Test loss: {loss:.4f}')
print(f'Test accuracy: {acc:.4f}')

test_probs = model.predict(test_ds, verbose=1)
test_pred = np.argmax(test_probs, axis=1)
test_true = test_df['label'].to_numpy()

print(classification_report(test_true, test_pred, labels=[0,1,2], target_names=CLASS_NAMES, digits=4, zero_division=0))

cm = confusion_matrix(test_true, test_pred, labels=[0,1,2])
fig, ax = plt.subplots(figsize=(7,6))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, cmap='Blues', values_format='d', colorbar=False)
plt.title('Test Confusion Matrix')
plt.tight_layout(); plt.savefig(OUTPUT_DIR/'confusion_matrix.png', dpi=170); plt.show()

no_idx = CLASS_TO_INDEX['NO_MAIZE']
actual_no = test_true == no_idx
wrong_as_maize = (test_pred != no_idx) & actual_no
print('NO_MAIZE test images:', int(actual_no.sum()))
print('NO_MAIZE predicted as maize:', int(wrong_as_maize.sum()))
print('Raw NO_MAIZE false-activation rate:', f'{wrong_as_maize.sum()/max(actual_no.sum(),1):.4%}')

## Validation-based sorting thresholds

Good and bad maize get separate confidence/margin thresholds. The search requires
at least three accepted validation examples, at least 95% empirical precision,
and at most 1% of validation `NO_MAIZE` examples accepted as either seed class.
No feasible policy means **actuation is disabled** in the exported metadata;
there is no fallback to an unsafe threshold. These are validation targets, not
promises about unseen seeds. Small validation counts cannot establish a 1% real-world
error rate; uncertainty bounds and test errors are reported below.

In [ ]:
# 15. Calibrate class-specific policies using validation only.
val_probs = np.asarray(model.predict(val_ds, verbose=0), dtype=np.float64)
val_true = val_df['label'].to_numpy()
val_pred = np.argmax(val_probs, axis=1)
ordered = np.sort(val_probs, axis=1)
val_conf, val_margin = ordered[:, -1], ordered[:, -1] - ordered[:, -2]
MIN_ACCEPTED = 3
MIN_PRECISION = 0.95
MAX_NO_MAIZE_RATE = 0.01
rows, CLASS_THRESHOLDS, supported = [], {}, []
# Each seed class gets half the total NO_MAIZE false-acceptance budget.
for class_name in CLASS_NAMES[:2]:
    class_index = CLASS_TO_INDEX[class_name]
    candidates = []
    for conf_t in np.arange(0.80, 1.00, 0.01):
        for margin_t in np.arange(0.20, 0.91, 0.05):
            conf_t, margin_t = round(float(conf_t), 2), round(float(margin_t), 2)
            accepted = (val_pred == class_index) & (val_conf >= conf_t) & (val_margin >= margin_t)
            correct = int((accepted & (val_true == class_index)).sum())
            count = int(accepted.sum())
            false_no = int((accepted & (val_true == no_idx)).sum())
            row = {'class_name': class_name, 'confidence': float(round(conf_t, 2)), 'margin': float(round(margin_t, 2)),
                   'accepted': count, 'correct': correct, 'precision': correct / max(count, 1),
                   'no_maize_false_acceptance': false_no / max(int((val_true == no_idx).sum()), 1)}
            rows.append(row)
            if count >= MIN_ACCEPTED and row['precision'] >= MIN_PRECISION and row['no_maize_false_acceptance'] <= MAX_NO_MAIZE_RATE / 2:
                candidates.append(row)
    candidates.sort(key=lambda row: (-row['correct'], -row['precision'], -row['confidence'], -row['margin']))
    supported.append(bool(candidates))
    best = candidates[0] if candidates else {'confidence': 1.0, 'margin': 1.0}
    CLASS_THRESHOLDS[class_name] = {'confidence': best['confidence'], 'margin': best['margin']}
ACTUATION_ENABLED = bool(all(supported))
DECISION_POLICY = {'actuation_enabled': ACTUATION_ENABLED, 'class_thresholds': CLASS_THRESHOLDS,
                   'empty_confidence': 0.90, 'empty_margin': 0.30,
                   'validation_min_precision': MIN_PRECISION, 'validation_max_no_maize_rate': MAX_NO_MAIZE_RATE}
pd.DataFrame(rows).to_csv(OUTPUT_DIR/'threshold_search_results.csv', index=False)
print(json.dumps(DECISION_POLICY, indent=2))
if not ACTUATION_ENABLED:
    print('No acceptable policy for both seed classes: export remains usable for dry-run inference, with actuation DISABLED.')

def accepted_mask(probabilities):
    # Match Python runtime comparisons at threshold boundaries, including float32 outputs.
    probabilities = np.asarray(probabilities, dtype=np.float64)
    pred = probabilities.argmax(axis=1)
    ordered = np.sort(probabilities, axis=1)
    confidence, margin = ordered[:, -1], ordered[:, -1] - ordered[:, -2]
    accepted = np.zeros(len(pred), dtype=bool)
    if ACTUATION_ENABLED:
        for name in CLASS_NAMES[:2]:
            threshold = CLASS_THRESHOLDS[name]
            accepted |= (pred == CLASS_TO_INDEX[name]) & (confidence >= threshold['confidence']) & (margin >= threshold['margin'])
    return accepted

In [ ]:
# 16. Evaluate the FIXED policy on untouched test originals; never tune on test results.
def wilson_interval(successes, total, z=1.96):
    if total == 0:
        return [None, None]
    p = successes / total
    denominator = 1 + z*z / total
    center = (p + z*z/(2*total)) / denominator
    half = z * np.sqrt(p*(1-p)/total + z*z/(4*total*total)) / denominator
    return [max(0.0, float(center-half)), min(1.0, float(center+half))]

eligible = accepted_mask(test_probs)
actual_no = test_true == no_idx
actual_maize = ~actual_no
false_act = actual_no & eligible
wrong_quality = actual_maize & eligible & (test_pred != test_true)
correct_maize = actual_maize & eligible & (test_pred == test_true)
TEST_METRICS = {
    'accuracy': float(accuracy_score(test_true, test_pred)),
    'macro_f1': float(f1_score(test_true, test_pred, labels=[0,1,2], average='macro', zero_division=0)),
    'no_maize_count': int(actual_no.sum()), 'false_no_maize_actuations': int(false_act.sum()),
    'false_no_maize_rate_wilson95': wilson_interval(int(false_act.sum()), int(actual_no.sum())),
    'wrong_good_bad_actuations': int(wrong_quality.sum()),
    'accepted_maize': int((actual_maize & eligible).sum()),
    'correct_maize_accept_rate': float(correct_maize.sum()/max(actual_maize.sum(),1)),
    'class_report': classification_report(test_true, test_pred, labels=[0,1,2], target_names=CLASS_NAMES, zero_division=0, output_dict=True),
}
print(json.dumps(TEST_METRICS, indent=2))
with open(OUTPUT_DIR/'test_metrics.json', 'w') as f:
    json.dump(TEST_METRICS, f, indent=2)
predictions = test_df[['source_path', 'label_name']].copy()
predictions['prediction'] = [INDEX_TO_CLASS[i] for i in test_pred]
predictions['eligible'] = eligible
for i, name in enumerate(CLASS_NAMES):
    predictions[f'p_{name}'] = test_probs[:, i]
predictions.to_csv(OUTPUT_DIR/'test_predictions.csv', index=False)

In [ ]:
# 17. Save the selected inference model; metadata is finalized AFTER export verification.
FINAL_KERAS = OUTPUT_DIR/'maize_mobilenetv3large_final.keras'
inference_model.save(FINAL_KERAS)
metadata = {
    'schema_version': 2, 'run_id': RUN_ID, 'model_name': 'MobileNetV3Large',
    'image_size': list(IMAGE_SIZE), 'class_names': CLASS_NAMES, 'class_to_index': CLASS_TO_INDEX,
    'input_color_order': 'RGB', 'input_dtype': 'float32', 'external_input_range': [0.0, 255.0],
    'resize_method': 'pillow_bilinear',
    'external_preprocessing': 'EXIF-correct RGB; Pillow BILINEAR resize; float32 0..255. Scaling is embedded.',
    'decision_policy': DECISION_POLICY, 'selected_checkpoint': str(final_checkpoint),
    'checkpoint_comparison': checkpoint_scores,
    'serial_protocol': 'CRC16-CCITT/session/boot/sequence v1',
    'actuator_mapping': {'BAD_SEED': 'BAD', 'GOOD_SEED': 'GOOD', 'NO_MAIZE': None},
    'original_class_counts': df['label_name'].value_counts().to_dict(),
    'tensorflow_version': tf.__version__,
}
print('Saved:', FINAL_KERAS)

In [ ]:
# 18. Export with an explicit batch-1 RGB signature and built-in TFLite operations.
SAVED_MODEL_DIR = LOCAL_CACHE/'inference_saved_model'
class ExportModule(tf.Module):
    def __init__(self, core):
        super().__init__()
        self.core = core
    @tf.function(input_signature=[tf.TensorSpec([1, IMAGE_SIZE[0], IMAGE_SIZE[1], 3], tf.float32, name='image')])
    def serve(self, image):
        return {'probabilities': self.core(image, training=False)}
export_module = ExportModule(inference_model)
tf.saved_model.save(export_module, str(SAVED_MODEL_DIR), signatures={'serving_default': export_module.serve})
FLOAT_TFLITE = OUTPUT_DIR/'maize_mobilenetv3large_float32.tflite'
converter = tf.lite.TFLiteConverter.from_saved_model(str(SAVED_MODEL_DIR))
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
FLOAT_TFLITE.write_bytes(converter.convert())
DYNAMIC_TFLITE = OUTPUT_DIR/'maize_mobilenetv3large_dynamic.tflite'
converter = tf.lite.TFLiteConverter.from_saved_model(str(SAVED_MODEL_DIR))
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
converter.optimizations = [tf.lite.Optimize.DEFAULT]
DYNAMIC_TFLITE.write_bytes(converter.convert())
print('Saved:', FLOAT_TFLITE, DYNAMIC_TFLITE)

In [ ]:
# 19. Verify ALL test originals with the same Pillow preprocessing used by main.py.
def prepare_rgb(image, size=IMAGE_SIZE):
    image = ImageOps.exif_transpose(image).convert('RGB')
    return np.asarray(image.resize((size[1], size[0]), Image.Resampling.BILINEAR), dtype=np.float32)[None]

def tflite_predict(model_path, paths):
    interpreter = tf.lite.Interpreter(model_path=str(model_path), num_threads=2)
    interpreter.allocate_tensors()
    inp, out = interpreter.get_input_details()[0], interpreter.get_output_details()[0]
    assert list(inp['shape']) == [1, *IMAGE_SIZE, 3] and list(out['shape']) == [1, 3]
    assert inp['dtype'] == np.float32 and out['dtype'] == np.float32
    probabilities = []
    for path in tqdm(paths, desc=model_path.name):
        with Image.open(path) as image:
            array = prepare_rgb(image)
        interpreter.set_tensor(inp['index'], array)
        interpreter.invoke()
        probabilities.append(interpreter.get_tensor(out['index'])[0])
    return np.asarray(probabilities)

fp = tflite_predict(FLOAT_TFLITE, test_df['path'])
dp = tflite_predict(DYNAMIC_TFLITE, test_df['path'])
export_checks = {}
verified_files = []
for path, probabilities in [(FLOAT_TFLITE, fp), (DYNAMIC_TFLITE, dp)]:
    max_difference = float(np.max(np.abs(probabilities - test_probs)))
    class_agreement = float(np.mean(probabilities.argmax(axis=1) == test_pred))
    decision_agreement = bool(np.array_equal(accepted_mask(probabilities), accepted_mask(test_probs)))
    passed = bool(np.isfinite(probabilities).all() and max_difference <= 1e-3 and class_agreement == 1.0 and decision_agreement)
    export_checks[path.name] = {'accuracy': float(accuracy_score(test_true, probabilities.argmax(axis=1))),
                              'max_probability_difference': max_difference, 'class_agreement': class_agreement,
                              'decision_agreement': decision_agreement, 'passed': passed}
    if passed:
        verified_files.append(path)
print(json.dumps(export_checks, indent=2))
if FLOAT_TFLITE not in verified_files:
    raise RuntimeError('Float32 TFLite export differs from Keras. Do not deploy; inspect export_checks.')
metadata['export_checks'] = export_checks
metadata['tflite_sha256'] = {path.name: hashlib.sha256(path.read_bytes()).hexdigest() for path in verified_files}
with open(OUTPUT_DIR/'deployment_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
# Only verified models go into the downloadable deployment bundle.
import zipfile
BUNDLE_PATH = OUTPUT_DIR/'maize_deployment.zip'
with zipfile.ZipFile(BUNDLE_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in [*verified_files, OUTPUT_DIR/'deployment_metadata.json', OUTPUT_DIR/'test_metrics.json']:
        archive.write(path, arcname='model_outputs/' + path.name)
print('DEPLOYMENT BUNDLE:', BUNDLE_PATH)
print('Copy main.py separately from this project. Extract the ZIP alongside it.')

In [ ]:
# 20. Single-image inference helper (no physical actuation from Colab).
def predict_single_image(image_path, show=True):
    with Image.open(image_path) as source:
        img = ImageOps.exif_transpose(source).convert('RGB')
        arr = prepare_rgb(img)
    probs = inference_model.predict(arr, verbose=0)[0]
    pred = int(np.argmax(probs))
    ordered = np.sort(probs)
    name = INDEX_TO_CLASS[pred]
    eligible = bool(accepted_mask(probs[None])[0])
    result = {'predicted_class': name, 'confidence': float(ordered[-1]),
              'margin': float(ordered[-1] - ordered[-2]),
              'serial_class': metadata['actuator_mapping'][name] if eligible else None,
              'probabilities': {CLASS_NAMES[i]: float(probs[i]) for i in range(3)}}
    if show:
        plt.figure(figsize=(5,5)); plt.imshow(img); plt.axis('off'); plt.title(f'{name} | eligible={eligible}'); plt.show()
    print(json.dumps(result, indent=2))
    return result
# predict_single_image('/content/drive/MyDrive/MaizeData/GOOD_SEED/example.jpg')

# Raspberry Pi + ESP32 deployment

1. Run all cells and inspect `test_metrics.json`, the confusion matrix, and `test_predictions.csv`.
2. Download `maize_deployment.zip` from the printed run directory. Extract beside `main.py` on the Pi.
3. Upload `esp32_maize_sorter/esp32_maize_sorter.ino` to the ESP and follow the repository README for wiring.
4. Test `python main.py --dry-run` with an empty view, then a single seed, then an empty view again.
5. Run `python main.py --port /dev/serial/by-id/YOUR_ESP_DEVICE` to enable USB serial sorting.

The Pi checks the model checksum, class mapping and preprocessing, requires stable
predictions, and sends one acknowledged GOOD/BAD command per observed seed. The
ESP returns to neutral automatically. Uncertain or NO_MAIZE predictions do not sort.
Leave a visible empty gap between seeds; without a presence sensor, touching seeds
cannot be reliably distinguished. Servo angles, GPIO, and cycle timing are editable
at the top of the sketch. A 3–4 second seed visibility period assumes a continuous
camera stream, not one image every 3–4 seconds.

Keep backgrounds, seed size within the image, camera distance and lighting close
to training conditions. Include real hard negatives and separate recording sessions
in evaluation. Synthetic augmentation and a high model confidence do not establish
accuracy on unseen backgrounds or hidden/internal seed defects.